# Non-Graded (Optional) Practice Tutorial: Python, SQLAlchemy, SQLite, and Streamlit

## Prerequisites for Extension Installation and Testing:
Complete the set up for SQLTools in the notebook found in the File Browser. It will be title `START_HERE_SettingUpSQLTools.ipynb`

You can toggle the File Browser by selecting 'View' and then 'File Browser'.

## Demonstration/Exercise: Create Your Project Structure

1. In your computer’s folder system, create a project folder for this assignment.
    2. Ensure you can locate this project effectively when necessary
2. Make sure that your folder is named like the following:

    `FirstName_LastInitial_SQLAlchemy_Streamlit`

3. Within VSCode, use the terminal to `cd` into your project folder.
    1. You’ll arrive at the project’s “root” directory
4. Inside the root directory, use the following command to create a `.venv` virtual environment:

    `python -m venv .venv`

   1. Create a `.gitignore` file inside of your project folder.
   2. Inside the `.gitignore`, add the `.venv` and save the file before moving onward
   3. This will ensure that no extra files are pushed to Git, though the project can be cloned and the packages can still be installed

**What does `python -m venv .venv` do?**
* `python`: use the program called python
* `-m`: call a module as a script, we'll tell it which module next
* `venv`: use the module called venv that normally comes installed with Python
* `.venv`: create the virtual environment in the new directory .venv

***<u>You only need to do this once per project</ul>***, not every time you work.

That command creates a new virtual environment in a directory called `.venv`. You could create the virtual environment in a different directory, but there's a industry convention of calling it `.venv` to make it easily identifiable to other developers, should your project call for collaboration, either through trouble-shooting or shared development and responsibilities.

5. Activate the virtual environment:
    1. For Mac/Linux users:

        `source .venv/bin/activate`

   2.  For Windows users:
  
        `.venv\Scripts\Activate.ps1`
  
    3. Windows (CLI “Command Prompt):
  
        `.venv\Scripts\activate.bat`
  
Do this <u>***every time***</u> you **start a new terminal session** to work on the project.
  
6. Use the following command to install the required Python packages:

    `pip install sqlalchemy streamlit`

**Do this once** when installing or upgrading the packages your project needs. If you need to upgrade a version or add a new package, you would do this again.

8. Use the following command to save the packages installed to a `requirements.txt` file:

    `pip freeze > requirements.txt`

A `requirements.txt` with some packages could look like:

    sqlmodel==0.13.0
    rich==13.7.1

Your project folder structure should now look like this:

    FirstName_LastInitial_SQLAlchemy_Streamlit/
         .venv/

8. Open the project in VSCode:

    `code .`

   1. Alternatively, open VSCode, select `File`, then select `Open folder`, and lastly, select the project folder from your file browser

## Creating Models

We will now create:
1. A SQLite database
2. SQLAlchemy models
3. A database engine
4. A session factory

### Demonstration/Exercise: Create SQLAlchemy Models
1. Inside the project folder, create a file named:

    `models.py`

2. Add the following import statements:

In [ ]:
from sqlalchemy import Column, Integer, String, Boolean, DateTime, ForeignKey
from sqlalchemy.orm import declarative_base, relationship
from datetime import datetime

DBModelBase = declarative_base()

Here is what is going on in the following code above:
* Imports the SQLAlchemy column types used to define fields in database tables.
* Imports SQLAlchemy tools for creating ORM models and defining table relationships.
* Imports Python’s datetime so timestamps can be generated.
* Creates the base class that all ORM models will inherit from so SQLAlchemy knows they represent database tables.

3. Create the `User` model:

In [ ]:
class User(DBModelBase):
    __tablename__ = "users"

    id = Column(Integer, primary_key=True)
    email = Column(String(255), nullable=False)
    name = Column(String(255), nullable=False)
    role = Column(String(255), nullable=False)

    posts = relationship("Post", back_populates="author", cascade="all, delete-orphan")
    comments = relationship("Comment", back_populates="author", cascade="all, delete-orphan")

    created_at = Column(DateTime, nullable=False, default=datetime.now())
    updated_at = Column(DateTime, nullable=False, default=datetime.now())

Here is an explanation of what the user model is storing:
    
    id — The primary key for each user
    email — A required email field
    name — The user’s display name
    role — A string representing one of "admin", "editor", "user", determined by the seeder
    posts — A relationship showing that 1 user can have many posts
    comments — A relationship showing that 1 user can have many comments
    created_at / updated_at — Automatically set timestamps

4. Create the Post model:

In [ ]:
class Post(DBModelBase):
    __tablename__ = "posts"

    id = Column(Integer, primary_key=True)
    title = Column(String(255), nullable=False)
    content = Column(String, nullable=False)
    published = Column(Boolean, default=True, nullable=False)

    author_id = Column(Integer, ForeignKey("users.id"), nullable=False)

    author = relationship("User", back_populates="posts")
    comments = relationship("Comment", back_populates="post", cascade="all, delete-orphan")

    created_at = Column(DateTime, nullable=False, default=datetime.now())
    updated_at = Column(DateTime, nullable=False, default=datetime.now())

Here is an explanation of what the post model is collecting:

    id — The primary key for each user
    title / content — Required fields
    published — A boolean flag (True/False)
    author_id — Foreign key connecting to a User
    author — Access the user who wrote the comment
    comments — Access to comment(s) belonging to this post

5. Create the Comment model:

In [ ]:
class Comment(DBModelBase):
    __tablename__ = "comments"

    id = Column(Integer, primary_key=True)
    text = Column(String, nullable=False)

    author_id = Column(Integer, ForeignKey("users.id"), nullable=False)
    post_id = Column(Integer, ForeignKey("posts.id"), nullable=False)

    author = relationship("User", back_populates="comments")
    post = relationship("Post", back_populates="comments")

    created_at = Column(DateTime, nullable=False, default=datetime.now())
    updated_at = Column(DateTime, nullable=False, default=datetime.now())

Here is an explanation of what the post model is collecting:

    id — The primary key for each user
    text — Required comment text
    author_id / post_id — Foreign keys linking to User and Post
    author — Access the user who wrote the comment
    post — Access the post this comment belongs to

## Creating a Database Engine
### Demonstration/Exercise: SQLAlchemy

1. Inside the project folder, create a file named:

    `database.py`

2. Inside of `database.py`, add the following:

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from db.models import DBModelBase

DATABASE_URL = "sqlite:///test.db"

engine = create_engine(DATABASE_URL)

DBModelBase.metadata.create_all(engine)

SessionLocal = sessionmaker(bind=engine)

What Is This Doing?
* Creates/opens a SQLite file named test.db
* Creates tables based on your models
* Prepares SessionLocal() for use in the seeder and Streamlit app


Now it’s time to create the tables.

3. Run the following command to create the tables:

    `python run database.py`

This command will generate a new file called `test.db`. That is the local database that contains our tables.

## Seeding the Database
We will now create a seeder to populate:
* 10 users
* 3 posts per user
* 2 comments per post

### Demonstration/Exercise: Create the Seeder
1. Inside the `db` folder, create:

    `seed.py`

2. In `seed.py`, add the following:

In [ ]:
from faker import Faker
from sqlalchemy.orm import Session
from db.database import SessionLocal
from db.models import User, Post, Comment
from enum import Enum

fake = Faker()

class Role(Enum):
    USER = "user"
    EDITOR = "editor"
    ADMIN = "admin"

3. Below this code, add the role helper:

In [ ]:
def assign_user_role(i: int) -> str:
    if i % 2 == 0:
        if i % 3 == 0:
            return Role.ADMIN.value
        return Role.EDITOR.value
    return Role.USER.value

4. Below the role helper, add the seeder code:

In [ ]:
def seeder(session: Session) -> None:
    added_users = []

    # Create Users
    for i in range(10):
        user = User(
            email=fake.email(),
            name=fake.name(),
            role=assign_user_role(i),
        )
        session.add(user)
        added_users.append(user)

    session.commit()

    # Create Posts & Comments
    for user in added_users:

        new_posts = []
        for _ in range(3):
            post = Post(
                title=fake.sentence(),
                content=fake.paragraph(),
                author=user,
            )
            session.add(post)
            new_posts.append(post)

        session.flush()

        for post in new_posts:
            for _ in range(2):
                comment = Comment(
                    text=fake.sentence(),
                    author=user,
                    post=post,
                )
                session.add(comment)

    session.commit()
    print("Seeding complete!")

5. Add entrypoint for the session:

In [ ]:
if __name__ == "__main__":
    with SessionLocal() as session:
        seeder(session)

6. Run the seeder from the terminal:

    `python db/seed.py`

You should see:

    `Seeding complete!`

7. Check your database using your SQLite viewer.

## Building the Streamlit Frontend

### Demonstration/Exercise: Create the Streamlit App

1. Inside the app folder create:

    `app.py`

2. Add the imports at the top of `app.py`:

In [ ]:
import streamlit as st
from sqlalchemy.orm import selectinload
from db.database import SessionLocal
from db.models import User, Post, Comment

3. Create helper to load users with eager-loading:

In [ ]:
def get_users():
    with SessionLocal() as db:
        return (
            db.query(User)
            .options(selectinload(User.posts).selectinload(Post.comments))
            .all()
        )

4. Add Streamlit UI:

In [ ]:
def main():
    st.set_page_config(page_title="Blog Explorer", layout="wide")
    st.title("📰 Blog Explorer (SQLAlchemy + SQLite + Streamlit)")

    users = get_users()

    if not users:
        st.warning("No users found in the database. Run the seeder first.")
        st.stop()

    st.sidebar.header("Select a User")
    selected_user = st.sidebar.selectbox(
        "Users",
        users,
        format_func=lambda u: u.name,
    )

    st.subheader(f"Posts by {selected_user.name}")

    for post in selected_user.posts:
        with st.expander(post.title, expanded=False):
            st.write(post.content)

            comments = post.comments
            if comments:
                st.markdown("**Comments:**")
                for comment in comments:
                    st.markdown(f"- {comment.text}")
            else:
                st.markdown("_No comments for this post._")

5. Add entrypoint:

In [ ]:
if __name__ == "__main__":
    main()

## Running the Frontend

1. Run Streamlit from the terminal:

   `streamlit run app/app.py`

2. You should see:

3. Click a user in the sidebar

4. Expand a post.

## Summary of Achievements

You have successfully:
* Created SQLAlchemy models
* Created a SQLite database
* Built a seeder using Faker
* Connected a Streamlit app to a database
* Loaded users, posts, and comments with eager-loading
* Built interactive UI components (expanders, sidebars)
* Displayed relational data in a live frontend


This completes the full guided exercise in the style of the SQLTools/Prisma instructional document.
